# Singapore Tuition Demand Seasonality — Segmented Prediction Pipeline

Predicts when demand peaks for each tuition subject and level, so ad spend can be allocated per segment.

**11 segments:** Primary (Math, English, Science) · Secondary (Math, English, Biology, Chemistry, Physics) · JC (Math, Chemistry, Economics)

**Pipeline overview:**
1. Fetch Google Trends search volume as a demand proxy — one keyword per segment
2. Collect MOE school calendar and SEAB exam dates
3. Engineer calendar and exam-urgency features
4. Use Spark `applyInPandas` to train one Prophet model per segment in parallel
5. Produce a per-segment 12-month forecast and ad/promo calendar

In [ ]:
%pip install prophet pytrends requests pandas numpy matplotlib seaborn -q

In [ ]:
import time
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
from datetime import timedelta
from pytrends.request import TrendReq
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (16, 5)
sns.set_theme(style='whitegrid')
print('Imports successful')

## Section 1: Segments, MOE Calendar & SEAB Exam Dates

In [ ]:
# --- Segments: display name -> Google Trends keyword ---
SEGMENTS = {
    'Primary Math':        'primary math tuition singapore',
    'Primary English':     'primary english tuition singapore',
    'Primary Science':     'primary science tuition singapore',
    'Secondary Math':      'secondary math tuition singapore',
    'Secondary English':   'secondary english tuition singapore',
    'Secondary Biology':   'secondary biology tuition singapore',
    'Secondary Chemistry': 'secondary chemistry tuition singapore',
    'Secondary Physics':   'secondary physics tuition singapore',
    'JC Math':             'jc math tuition singapore',
    'JC Chemistry':        'jc chemistry tuition singapore',
    'JC Economics':        'jc economics tuition singapore',
}

SEG_ORDER = [
    'Primary Math','Primary English','Primary Science',
    'Secondary Math','Secondary English','Secondary Biology','Secondary Chemistry','Secondary Physics',
    'JC Math','JC Chemistry','JC Economics',
]

# --- MOE school term and holiday dates 2020-2026 ---
# Source: https://www.moe.gov.sg/news/press-releases
MOE_CALENDAR = {
    2020: {
        'term1': ('2020-01-02','2020-03-13'), 'term2': ('2020-03-23','2020-05-29'),
        'term3': ('2020-06-29','2020-09-04'), 'term4': ('2020-09-14','2020-11-20'),
        'march_hols': ('2020-03-14','2020-03-22'), 'june_hols': ('2020-05-30','2020-06-28'),
        'sept_hols': ('2020-09-05','2020-09-13'),  'year_end':  ('2020-11-21','2020-12-31'),
    },
    2021: {
        'term1': ('2021-01-04','2021-03-12'), 'term2': ('2021-03-22','2021-05-28'),
        'term3': ('2021-06-28','2021-09-03'), 'term4': ('2021-09-13','2021-11-19'),
        'march_hols': ('2021-03-13','2021-03-21'), 'june_hols': ('2021-05-29','2021-06-27'),
        'sept_hols': ('2021-09-04','2021-09-12'),  'year_end':  ('2021-11-20','2021-12-31'),
    },
    2022: {
        'term1': ('2022-01-03','2022-03-11'), 'term2': ('2022-03-21','2022-05-27'),
        'term3': ('2022-06-27','2022-09-02'), 'term4': ('2022-09-12','2022-11-18'),
        'march_hols': ('2022-03-12','2022-03-20'), 'june_hols': ('2022-05-28','2022-06-26'),
        'sept_hols': ('2022-09-03','2022-09-11'),  'year_end':  ('2022-11-19','2022-12-31'),
    },
    2023: {
        'term1': ('2023-01-03','2023-03-10'), 'term2': ('2023-03-20','2023-05-26'),
        'term3': ('2023-06-26','2023-09-01'), 'term4': ('2023-09-11','2023-11-17'),
        'march_hols': ('2023-03-11','2023-03-19'), 'june_hols': ('2023-05-27','2023-06-25'),
        'sept_hols': ('2023-09-02','2023-09-10'),  'year_end':  ('2023-11-18','2023-12-31'),
    },
    2024: {
        'term1': ('2024-01-02','2024-03-08'), 'term2': ('2024-03-18','2024-05-24'),
        'term3': ('2024-06-24','2024-08-30'), 'term4': ('2024-09-09','2024-11-15'),
        'march_hols': ('2024-03-09','2024-03-17'), 'june_hols': ('2024-05-25','2024-06-23'),
        'sept_hols': ('2024-08-31','2024-09-08'),  'year_end':  ('2024-11-16','2024-12-31'),
    },
    2025: {
        'term1': ('2025-01-02','2025-03-14'), 'term2': ('2025-03-24','2025-05-30'),
        'term3': ('2025-06-30','2025-09-05'), 'term4': ('2025-09-15','2025-11-21'),
        'march_hols': ('2025-03-15','2025-03-23'), 'june_hols': ('2025-05-31','2025-06-29'),
        'sept_hols': ('2025-09-06','2025-09-14'),  'year_end':  ('2025-11-22','2025-12-31'),
    },
    2026: {
        'term1': ('2026-01-02','2026-03-13'), 'term2': ('2026-03-23','2026-05-29'),
        'term3': ('2026-06-29','2026-09-04'), 'term4': ('2026-09-14','2026-11-20'),
        'march_hols': ('2026-03-14','2026-03-22'), 'june_hols': ('2026-05-30','2026-06-28'),
        'sept_hols': ('2026-09-05','2026-09-13'),  'year_end':  ('2026-11-21','2026-12-31'),
    },
}

# --- SEAB exam and results dates ---
# Source: https://www.seab.gov.sg/important-dates-for-candidates
SEAB_EVENTS = [
    {'event':'SA1_exam',       'start':'2020-04-27','end':'2020-05-08'},
    {'event':'SA1_exam',       'start':'2021-04-26','end':'2021-05-07'},
    {'event':'SA1_exam',       'start':'2022-04-25','end':'2022-05-06'},
    {'event':'SA1_exam',       'start':'2023-04-24','end':'2023-05-05'},
    {'event':'SA1_exam',       'start':'2024-04-22','end':'2024-05-03'},
    {'event':'SA1_exam',       'start':'2025-04-28','end':'2025-05-09'},
    {'event':'SA1_exam',       'start':'2026-04-27','end':'2026-05-08'},
    {'event':'SA2_exam',       'start':'2020-09-28','end':'2020-10-16'},
    {'event':'SA2_exam',       'start':'2021-09-27','end':'2021-10-15'},
    {'event':'SA2_exam',       'start':'2022-09-26','end':'2022-10-14'},
    {'event':'SA2_exam',       'start':'2023-09-25','end':'2023-10-13'},
    {'event':'SA2_exam',       'start':'2024-09-23','end':'2024-10-11'},
    {'event':'SA2_exam',       'start':'2025-09-29','end':'2025-10-17'},
    {'event':'SA2_exam',       'start':'2026-09-28','end':'2026-10-16'},
    {'event':'PSLE_exam',      'start':'2020-08-31','end':'2020-09-25'},
    {'event':'PSLE_exam',      'start':'2021-08-30','end':'2021-09-24'},
    {'event':'PSLE_exam',      'start':'2022-09-01','end':'2022-09-29'},
    {'event':'PSLE_exam',      'start':'2023-08-31','end':'2023-09-28'},
    {'event':'PSLE_exam',      'start':'2024-08-27','end':'2024-09-27'},
    {'event':'PSLE_exam',      'start':'2025-08-25','end':'2025-09-26'},
    {'event':'PSLE_exam',      'start':'2026-08-12','end':'2026-09-30'},
    {'event':'PSLE_results',   'start':'2020-11-25','end':'2020-11-25'},
    {'event':'PSLE_results',   'start':'2021-11-24','end':'2021-11-24'},
    {'event':'PSLE_results',   'start':'2022-11-23','end':'2022-11-23'},
    {'event':'PSLE_results',   'start':'2023-11-22','end':'2023-11-22'},
    {'event':'PSLE_results',   'start':'2024-11-27','end':'2024-11-27'},
    {'event':'PSLE_results',   'start':'2025-11-26','end':'2025-11-26'},
    {'event':'PSLE_results',   'start':'2026-11-24','end':'2026-11-25'},
    {'event':'OLevel_exam',    'start':'2020-10-05','end':'2020-11-06'},
    {'event':'OLevel_exam',    'start':'2021-10-04','end':'2021-11-05'},
    {'event':'OLevel_exam',    'start':'2022-10-03','end':'2022-11-04'},
    {'event':'OLevel_exam',    'start':'2023-10-02','end':'2023-11-03'},
    {'event':'OLevel_exam',    'start':'2024-10-07','end':'2024-11-08'},
    {'event':'OLevel_exam',    'start':'2025-10-06','end':'2025-11-07'},
    {'event':'OLevel_exam',    'start':'2026-10-05','end':'2026-11-06'},
    {'event':'OLevel_results', 'start':'2021-01-11','end':'2021-01-11'},
    {'event':'OLevel_results', 'start':'2022-01-12','end':'2022-01-12'},
    {'event':'OLevel_results', 'start':'2023-01-11','end':'2023-01-11'},
    {'event':'OLevel_results', 'start':'2024-01-10','end':'2024-01-10'},
    {'event':'OLevel_results', 'start':'2025-01-14','end':'2025-01-14'},
    {'event':'OLevel_results', 'start':'2026-01-13','end':'2026-01-15'},
    {'event':'ALevel_exam',    'start':'2020-10-08','end':'2020-11-20'},
    {'event':'ALevel_exam',    'start':'2021-10-07','end':'2021-11-19'},
    {'event':'ALevel_exam',    'start':'2022-10-06','end':'2022-11-18'},
    {'event':'ALevel_exam',    'start':'2023-10-05','end':'2023-11-17'},
    {'event':'ALevel_exam',    'start':'2024-10-10','end':'2024-11-22'},
    {'event':'ALevel_exam',    'start':'2025-10-09','end':'2025-11-21'},
    {'event':'ALevel_exam',    'start':'2026-10-08','end':'2026-11-27'},
    {'event':'ALevel_results', 'start':'2021-02-26','end':'2021-02-26'},
    {'event':'ALevel_results', 'start':'2022-02-25','end':'2022-02-25'},
    {'event':'ALevel_results', 'start':'2023-02-24','end':'2023-02-24'},
    {'event':'ALevel_results', 'start':'2024-02-22','end':'2024-02-22'},
    {'event':'ALevel_results', 'start':'2025-02-21','end':'2025-02-21'},
    {'event':'ALevel_results', 'start':'2026-02-19','end':'2026-02-23'},
]

df_events = pd.DataFrame(SEAB_EVENTS)
df_events['start'] = pd.to_datetime(df_events['start'])
df_events['end']   = pd.to_datetime(df_events['end'])
print(f'{len(SEGMENTS)} segments defined')
print(f'{len(df_events)} exam/results events loaded')

## Section 2: Google Trends Data — Load from CSV Files

Monthly search volume data downloaded from [trends.google.com](https://trends.google.com) (region: Singapore, 2020–present).
7 unique keywords are loaded and mapped to 11 segments — segments at different levels that share a subject
(e.g. Primary Math, Secondary Math, JC Math) use the same Trends signal but get different exam urgency regressors.

In [ ]:
BASE_PATH = '/Volumes/workspace/default/trends_data'

# Keyword -> actual filename on Databricks
KEYWORD_FILES = {
    'math tuition':      'time_series_SG_20200101-0000_20260511-1054.csv',
    'english tuition':   'time_series_SG_20200101-0000_20260511-1054-2.csv',
    'science tuition':   'time_series_SG_20200101-0000_20260511-1055.csv',
    'biology tuition':   'time_series_SG_20200101-0000_20260511-1055-2.csv',
    'chemistry tuition': 'time_series_SG_20200101-0000_20260511-1056.csv',
    'physics tuition':   'time_series_SG_20200101-0000_20260511-1056-2.csv',
    'economics tuition': 'time_series_SG_20200101-0000_20260511-1056-3.csv',
}

# Segment -> keyword (multiple segments share a keyword, differentiated by exam urgency regressors)
SEGMENT_KEYWORDS = {
    'Primary Math':        'math tuition',
    'Primary English':     'english tuition',
    'Primary Science':     'science tuition',
    'Secondary Math':      'math tuition',
    'Secondary English':   'english tuition',
    'Secondary Biology':   'biology tuition',
    'Secondary Chemistry': 'chemistry tuition',
    'Secondary Physics':   'physics tuition',
    'JC Math':             'math tuition',
    'JC Chemistry':        'chemistry tuition',
    'JC Economics':        'economics tuition',
}

# Load each unique keyword CSV once
keyword_data = {}
for keyword, filename in KEYWORD_FILES.items():
    path = f'{BASE_PATH}/{filename}'
    df_tmp = pd.read_csv(path, parse_dates=['Time'])
    df_tmp.columns = ['date', 'value']
    df_tmp['value'] = pd.to_numeric(df_tmp['value'], errors='coerce').fillna(0)
    # Filter to 2020 onwards
    df_tmp = df_tmp[df_tmp['date'] >= '2020-01-01']
    keyword_data[keyword] = df_tmp.set_index('date')['value']
    print(f'Loaded: {keyword} ({len(df_tmp)} monthly rows, {df_tmp.value.eq(0).mean()*100:.0f}% zeros)')

# Build wide DataFrame — one column per segment
frames = {seg: keyword_data[kw].rename(seg) for seg, kw in SEGMENT_KEYWORDS.items()}
df_trends_wide = pd.DataFrame(frames).reset_index()
df_trends_wide.columns.name = None
df_trends_wide['date'] = pd.to_datetime(df_trends_wide['date'])
seg_cols = [c for c in df_trends_wide.columns if c != 'date']

print(f'\nTrends table: {len(df_trends_wide)} monthly rows x {len(seg_cols)} segments')
display(spark.createDataFrame(df_trends_wide.head(5).astype(str)))

### Bronze Layer — Raw Ingestion

Land source data into Delta tables with no transformation.
Bronze is the audit trail — if anything breaks downstream, we reprocess from here.

In [ ]:
CATALOG = 'workspace'
SCHEMA  = 'default'

# Raw Google Trends data (wide format, as downloaded)
(spark.createDataFrame(df_trends_wide.astype(str))
     .write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(f'{CATALOG}.{SCHEMA}.bronze_trends'))

# Raw SEAB exam and results events (reference data)
(spark.createDataFrame(pd.DataFrame(SEAB_EVENTS))
     .write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(f'{CATALOG}.{SCHEMA}.bronze_seab_events'))

print('Bronze tables written:')
print(f'  {CATALOG}.{SCHEMA}.bronze_trends        — {len(df_trends_wide)} rows')
print(f'  {CATALOG}.{SCHEMA}.bronze_seab_events   — {len(SEAB_EVENTS)} rows')

## Section 3: Build Segmented Demand Table

Reshape wide monthly Trends table into long format (one row per date x segment),
then resample from monthly to daily via forward fill.

In [ ]:
# Monthly -> daily via forward fill
df_daily_wide = (
    df_trends_wide
    .set_index('date')
    .resample('D')
    .ffill()
    .reset_index()
)

# Wide -> long
df_long = df_daily_wide.melt(
    id_vars='date',
    value_vars=seg_cols,
    var_name='segment',
    value_name='demand'
).dropna(subset=['demand'])

df_long['date']  = pd.to_datetime(df_long['date'])
df_long['level'] = df_long['segment'].str.split().str[0]  # Primary / Secondary / JC

print(f'Long-format table: {len(df_long):,} rows, {df_long.segment.nunique()} segments')
print(f'Date range: {df_long.date.min().date()} to {df_long.date.max().date()}')
display(spark.createDataFrame(df_long.head(20).astype(str)))

### Silver Layer — Validated & Cleaned

Read from Bronze, apply data quality checks, reshape to long format.
Rows that fail checks are flagged — the pipeline raises an error rather than silently producing bad output.

In [ ]:
# --- Data quality checks ---
assert df_long['date'].isna().sum() == 0,     'Silver check failed: null dates detected'
assert df_long['demand'].between(0, 100).all(),     'Silver check failed: demand values outside valid 0-100 range'
assert df_long['segment'].nunique() == 11,     f'Silver check failed: expected 11 segments, got {df_long["segment"].nunique()}'

# Warn if any segment has a date gap > 60 days (missing months)
for seg, grp in df_long.groupby('segment'):
    max_gap = grp['date'].sort_values().diff().dt.days.dropna().max()
    if max_gap > 60:
        print(f'WARNING: {seg} has a date gap of {int(max_gap)} days')

print('Silver quality checks passed.')

# Write to Silver Delta table
(spark.createDataFrame(df_long)
     .write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(f'{CATALOG}.{SCHEMA}.silver_demand'))

print(f'Silver table written: {CATALOG}.{SCHEMA}.silver_demand ({len(df_long):,} rows)')

## Section 4: Feature Engineering

Add calendar and exam-urgency features to every row.

Urgency features are level-specific:
- `primary_urgency` — days to PSLE
- `secondary_urgency` — days to O-Level
- `jc_urgency` — days to A-Level
- `sa_urgency` — days to SA1/SA2 (all levels)
- `results_urgency` — days since most recent results release (all levels)

In [ ]:
def build_urgency_series(dates, event_keyword, decay_days=14, direction='future'):
    import re
    filtered = [e for e in SEAB_EVENTS if re.search(event_keyword, e['event'])]
    starts   = [pd.to_datetime(e['start']) for e in filtered]
    scores   = []
    for d in dates:
        d = pd.Timestamp(d)
        if direction == 'future':
            upcoming = [s for s in starts if s >= d]
            delta = (min(upcoming) - d).days if upcoming else 365
        else:
            past = [s for s in starts if s <= d]
            delta = (d - max(past)).days if past else 365
        scores.append(float(np.exp(-delta / decay_days)))
    return scores

def is_school_holiday(d):
    cal = MOE_CALENDAR.get(d.year, {})
    for period in ['march_hols','june_hols','sept_hols','year_end']:
        if period in cal:
            s, e = pd.to_datetime(cal[period][0]), pd.to_datetime(cal[period][1])
            if s <= d <= e:
                return 1
    return 0

# Compute on unique dates only (much faster than row-by-row on the full long table)
unique_dates = df_long['date'].drop_duplicates().sort_values().reset_index(drop=True)

urgency_df = pd.DataFrame({'date': unique_dates})
urgency_df['primary_urgency']   = build_urgency_series(unique_dates, 'PSLE_exam')
urgency_df['secondary_urgency'] = build_urgency_series(unique_dates, 'OLevel_exam')
urgency_df['jc_urgency']        = build_urgency_series(unique_dates, 'ALevel_exam')
urgency_df['sa_urgency']        = build_urgency_series(unique_dates, 'SA[12]_exam')
urgency_df['results_urgency']   = build_urgency_series(unique_dates, 'results', direction='past')
urgency_df['is_school_holiday'] = unique_dates.apply(is_school_holiday).values

df_features = df_long.merge(urgency_df, on='date', how='left')
print(f'Feature table: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns')
display(spark.createDataFrame(df_features.head(10).astype(str)))

### Gold Layer — Feature-Engineered, Model-Ready

Enrich Silver data with all features. Gold is the table the model consumes.
Writing to Delta here means the model can always be retrained from Gold without re-running ingestion.

In [ ]:
# Write feature table to Gold Delta
(spark.createDataFrame(df_features)
     .write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(f'{CATALOG}.{SCHEMA}.gold_features'))

# Read back from Gold — model always consumes from this table
df_features = spark.read.table(f'{CATALOG}.{SCHEMA}.gold_features').toPandas()
df_features['date'] = pd.to_datetime(df_features['date'])

print(f'Gold table written and read back: {CATALOG}.{SCHEMA}.gold_features ({len(df_features):,} rows)')
display(spark.read.table(f'{CATALOG}.{SCHEMA}.gold_features').limit(5))

## Section 5: Exploratory Data Analysis

In [ ]:
# Monthly demand trend lines per level
monthly = (
    df_features
    .assign(month=lambda x: x['date'].dt.month)
    .groupby(['level','segment','month'])['demand']
    .mean()
    .reset_index()
)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, lvl in zip(axes, ['Primary','Secondary','JC']):
    for seg in [s for s in SEG_ORDER if s.startswith(lvl)]:
        sub = monthly[(monthly['level'] == lvl) & (monthly['segment'] == seg)]
        ax.plot(sub['month'], sub['demand'], marker='o', markersize=4,
                label=seg.replace(lvl+' ',''))
    ax.set_title(f'{lvl}')
    ax.set_xticks(range(1,13))
    ax.set_xticklabels(month_labels, rotation=45, fontsize=8)
    ax.set_ylabel('Avg Search Volume')
    ax.legend(fontsize=8)

plt.suptitle('Average Monthly Demand by Segment', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: all segments x month
pivot = (
    df_features
    .assign(month=lambda x: x['date'].dt.month)
    .groupby(['segment','month'])['demand']
    .mean()
    .unstack()
)
pivot.columns = month_labels
pivot = pivot.reindex(SEG_ORDER)

plt.figure(figsize=(16, 6))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5)
plt.title('Average Monthly Search Demand — All Segments')
plt.ylabel('')
plt.tight_layout()
plt.show()

## Section 6: Parallel Prophet Models via Spark

`applyInPandas` distributes one Prophet model per segment across Spark workers — all 11 train simultaneously.

Each model uses level-specific urgency regressors so Primary models respond to PSLE signals,
Secondary models to O-Level signals, and JC models to A-Level signals.

In [ ]:
# Build Prophet holidays DataFrame (captured as closure inside applyInPandas)
def build_sg_holidays(events_df):
    rows = []
    for _, row in events_df.iterrows():
        d = row['start']
        while d <= row['end']:
            rows.append({'holiday': row['event'], 'ds': d,
                         'lower_window': -3, 'upper_window': 14})
            d += timedelta(days=1)
    return pd.DataFrame(rows)

SG_HOLIDAYS = build_sg_holidays(df_events)

# Add school holiday periods
hol_dates = df_features[df_features['is_school_holiday'] == 1]['date'].drop_duplicates()
school_rows = [{'holiday': 'school_holiday', 'ds': d, 'lower_window': 0, 'upper_window': 0}
               for d in hol_dates]
SG_HOLIDAYS = pd.concat([SG_HOLIDAYS, pd.DataFrame(school_rows)], ignore_index=True)

FORECAST_DAYS = 365
print(f'Holidays table ready: {SG_HOLIDAYS.holiday.nunique()} distinct events')

In [ ]:
# Output schema for applyInPandas
FORECAST_SCHEMA = StructType([
    StructField('segment',    StringType()),
    StructField('ds',         TimestampType()),
    StructField('yhat',       DoubleType()),
    StructField('yhat_lower', DoubleType()),
    StructField('yhat_upper', DoubleType()),
    StructField('trend',      DoubleType()),
    StructField('yearly',     DoubleType()),
])


def _urgency_for_dates(dates, event_keyword, seab_events, decay_days=14, direction='future'):
    """Standalone urgency helper — no external DataFrame dependency."""
    import re
    starts = [pd.to_datetime(e['start']) for e in seab_events
              if re.search(event_keyword, e['event'])]
    scores = []
    for d in pd.to_datetime(dates):
        if direction == 'future':
            upcoming = [s for s in starts if s >= d]
            delta = (min(upcoming) - d).days if upcoming else 365
        else:
            past = [s for s in starts if s <= d]
            delta = (d - max(past)).days if past else 365
        scores.append(float(np.exp(-delta / decay_days)))
    return scores


def train_and_forecast(key, pdf):
    """Trains one Prophet model for a single segment. Runs on a Spark worker."""
    import warnings
    import pandas as pd
    import numpy as np
    from prophet import Prophet
    warnings.filterwarnings('ignore')

    segment = key[0]
    level   = segment.split()[0]  # 'Primary', 'Secondary', or 'JC'

    pdf = (pdf.rename(columns={'date': 'ds', 'demand': 'y'})
              .sort_values('ds')
              .dropna(subset=['y'])
              .reset_index(drop=True))
    pdf['ds'] = pd.to_datetime(pdf['ds'])

    empty = pd.DataFrame(columns=['segment','ds','yhat','yhat_lower','yhat_upper','trend','yearly'])
    if len(pdf) < 52:
        return empty

    # Level-specific main urgency column and exam keyword
    level_cfg = {
        'Primary':   ('primary_urgency',   'PSLE_exam'),
        'Secondary': ('secondary_urgency', 'OLevel_exam'),
        'JC':        ('jc_urgency',        'ALevel_exam'),
    }
    main_col, main_kw = level_cfg.get(level, ('sa_urgency', 'SA[12]_exam'))
    regressors = list(dict.fromkeys([main_col, 'sa_urgency', 'results_urgency']))
    regressors = [r for r in regressors if r in pdf.columns]

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        holidays=SG_HOLIDAYS,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.1,
        holidays_prior_scale=10.0,
        uncertainty_samples=200,
    )
    for r in regressors:
        m.add_regressor(r, standardize=True)

    m.fit(pdf[['ds', 'y'] + regressors])

    future = m.make_future_dataframe(periods=FORECAST_DAYS, freq='D')
    future[main_col]          = _urgency_for_dates(future['ds'], main_kw,   SEAB_EVENTS)
    future['sa_urgency']      = _urgency_for_dates(future['ds'], 'SA[12]_exam', SEAB_EVENTS)
    future['results_urgency'] = _urgency_for_dates(future['ds'], 'results', SEAB_EVENTS, direction='past')

    forecast = m.predict(future)
    yearly_col = 'yearly' if 'yearly' in forecast.columns else None
    out = forecast[['ds','yhat','yhat_lower','yhat_upper','trend'] +
                   ([yearly_col] if yearly_col else [])].copy()
    if yearly_col is None:
        out['yearly'] = 0.0
    out['yhat']       = out['yhat'].clip(0)
    out['yhat_lower'] = out['yhat_lower'].clip(0)
    out['segment']    = segment
    return out[['segment','ds','yhat','yhat_lower','yhat_upper','trend','yearly']]


print('applyInPandas function defined.')

In [ ]:
# Load feature table into Spark and run all 11 models in parallel
feature_sdf = spark.createDataFrame(df_features[[
    'date','segment','demand',
    'primary_urgency','secondary_urgency','jc_urgency',
    'sa_urgency','results_urgency','is_school_holiday'
]])

forecast_sdf = feature_sdf.groupby('segment').applyInPandas(train_and_forecast, schema=FORECAST_SCHEMA)

# Materialise (triggers the parallel training)
df_forecast = forecast_sdf.toPandas()
df_forecast['ds'] = pd.to_datetime(df_forecast['ds'])

print(f'Forecast complete: {len(df_forecast):,} rows across {df_forecast.segment.nunique()} segments')
display(forecast_sdf.orderBy('segment','ds').limit(20))

## Section 7: Cross-Validation (Primary Math)

Validate forecast accuracy on the highest-signal segment using rolling cross-validation.

In [ ]:
CV_SEGMENT = 'Primary Math'

pdf_cv = (df_features[df_features['segment'] == CV_SEGMENT]
          [['date','demand','primary_urgency','sa_urgency','results_urgency']]
          .rename(columns={'date':'ds','demand':'y'})
          .sort_values('ds'))

m_cv = Prophet(
    yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
    holidays=SG_HOLIDAYS, seasonality_mode='multiplicative',
    changepoint_prior_scale=0.1, holidays_prior_scale=10.0,
)
for r in ['primary_urgency','sa_urgency','results_urgency']:
    m_cv.add_regressor(r, standardize=True)
m_cv.fit(pdf_cv)

df_cv   = cross_validation(m_cv, initial='548 days', period='30 days', horizon='90 days', parallel='threads')
df_perf = performance_metrics(df_cv)
print(f'Cross-validation — {CV_SEGMENT}:')
print(df_perf[['horizon','mape','rmse','mae']].to_string(index=False))

fig = plot_cross_validation_metric(df_cv, metric='mape')
plt.title(f'MAPE by Forecast Horizon — {CV_SEGMENT}')
plt.tight_layout()
plt.show()

## Section 8: 12-Month Forecast & Per-Segment Ad Calendar

In [ ]:
last_hist = df_features['date'].max()
df_future = df_forecast[df_forecast['ds'] > last_hist].copy()

# Weekly aggregation per segment
df_weekly = (
    df_future
    .groupby('segment')
    .apply(lambda x: x.set_index('ds')[['yhat','yhat_lower','yhat_upper']].resample('W').sum())
    .reset_index()
)

# Demand tiers relative to each segment's own distribution
def assign_tiers(grp):
    p33, p67 = grp['yhat'].quantile(0.33), grp['yhat'].quantile(0.67)
    grp = grp.copy()
    grp['tier'] = pd.cut(grp['yhat'], bins=[-np.inf,p33,p67,np.inf],
                         labels=['LOW','MEDIUM','HIGH'])
    return grp

df_weekly = df_weekly.groupby('segment', group_keys=False).apply(assign_tiers)

# Plot per level
tier_colors = {'HIGH':'#2ca02c','MEDIUM':'#ff7f0e','LOW':'#d62728'}
from matplotlib.patches import Patch

for lvl in ['Primary','Secondary','JC']:
    segs = [s for s in SEG_ORDER if s.startswith(lvl)]
    fig, axes = plt.subplots(1, len(segs), figsize=(6*len(segs), 4))
    if len(segs) == 1:
        axes = [axes]
    for ax, seg in zip(axes, segs):
        sub = df_weekly[df_weekly['segment'] == seg]
        for tier, color in tier_colors.items():
            mask = sub['tier'] == tier
            ax.bar(sub['ds'][mask], sub['yhat'][mask], width=6, color=color, alpha=0.85)
        ax.fill_between(sub['ds'], sub['yhat_lower'], sub['yhat_upper'], alpha=0.12, color='grey')
        ax.set_title(seg.replace(lvl+' ',''), fontsize=10)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        plt.setp(ax.get_xticklabels(), rotation=45, fontsize=7)

    legend_els = [Patch(color=c, label=t) for t, c in tier_colors.items()]
    fig.legend(handles=legend_els, loc='upper right', fontsize=8)
    fig.suptitle(f'{lvl} — 12-Month Demand Forecast', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Segment x Month heatmap — the actionable ad calendar
df_weekly['month'] = df_weekly['ds'].dt.to_period('M').astype(str)
tier_num_map = {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3}

# Cast tier to string first to strip Categorical dtype, then map to numeric
df_weekly['tier_num'] = pd.to_numeric(df_weekly['tier'].astype(str).map(tier_num_map))

pivot_tier = (
    df_weekly.groupby(['segment','month'])['tier_num']
    .mean().unstack().reindex(SEG_ORDER)
)

from matplotlib.colors import ListedColormap
cmap = ListedColormap(['#d62728','#ff7f0e','#2ca02c'])

plt.figure(figsize=(16, 6))
sns.heatmap(pivot_tier, cmap=cmap, linewidths=0.5, vmin=1, vmax=3,
            annot=False, cbar_kws={'ticks': [1, 2, 3], 'label': 'LOW / MEDIUM / HIGH'})
plt.title('Ad/Promo Calendar — Demand Tier by Segment and Month\n'
          '(Green = push ads  ·  Red = run promos)')
plt.ylabel('')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Printable HIGH-demand weeks per segment
high_weeks = (
    df_weekly[df_weekly['tier'] == 'HIGH'][['segment','ds','yhat']]
    .rename(columns={'ds':'week_start','yhat':'projected_demand'})
    .sort_values(['segment','week_start'])
)
high_weeks['week_start']        = high_weeks['week_start'].dt.date
high_weeks['projected_demand']  = high_weeks['projected_demand'].round(1)

print('=== HIGH DEMAND WINDOWS — maximise ad spend ===')
for seg in SEG_ORDER:
    sub = high_weeks[high_weeks['segment'] == seg]
    if not sub.empty:
        weeks = ', '.join(sub['week_start'].astype(str).tolist())
        print(f'  {seg:26s}: {weeks}')

display(spark.createDataFrame(high_weeks))

In [ ]:
# Write forecast to Gold Delta table
(forecast_sdf
     .write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(f'{CATALOG}.{SCHEMA}.gold_forecast'))

print(f'Forecast saved to {CATALOG}.{SCHEMA}.gold_forecast')
print()
print('Lakehouse tables written this run:')
for tbl in ['bronze_trends','bronze_seab_events','silver_demand','gold_features','gold_forecast']:
    count = spark.read.table(f'{CATALOG}.{SCHEMA}.{tbl}').count()
    print(f'  {CATALOG}.{SCHEMA}.{tbl:30s} {count:>6} rows')